# Programmatic Optuna Tuning with VAMOS

This notebook shows a full **programmatic** tuning workflow built on VAMOS' algorithm config spaces and the Optuna backend.

We will:
1. start from `build_nsgaii_config_space()`,
2. run an Optuna study across multiple training problems,
3. export the history to JSON and CSV,
4. validate the tuned configuration against a default NSGA-II setup.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vamos import optimize
from vamos.algorithms import NSGAIIConfig
from vamos.engine.tuning import (
    EvalContext,
    Instance,
    ModelBasedTuner,
    TuningTask,
    available_model_based_backends,
    build_nsgaii_config_space,
    config_from_assignment,
    history_to_dict,
    save_history_csv,
    save_history_json,
)
from vamos.foundation.quality_indicators import compute_hypervolume

if not available_model_based_backends().get('optuna'):
    raise RuntimeError("Optuna is not available. Install vamos-optimization[tuning].")


## 1. Build the Search Space and Task

Here we rely on VAMOS' maintained NSGA-II config space instead of hand-coding the parameter definitions.


In [ ]:
REF_POINT = np.array([1.1, 1.1])
N_VAR = 20
TRAIN_BUDGET = 1500

config_space = build_nsgaii_config_space()
param_space = config_space.to_param_space()

instances = [
    Instance(name='zdt1', n_var=N_VAR, kwargs={}),
    Instance(name='zdt2', n_var=N_VAR, kwargs={}),
]


def eval_fn(config: dict, ctx: EvalContext) -> float:
    cfg = config_from_assignment('nsgaii', dict(config))
    result = optimize(
        ctx.instance.name,
        algorithm='nsgaii',
        algorithm_config=cfg,
        max_evaluations=ctx.budget,
        seed=ctx.seed,
        n_var=ctx.instance.n_var,
        engine='numba',
    )
    if result.F is None or len(result.F) == 0:
        return 0.0
    return float(compute_hypervolume(result.F, REF_POINT))


task = TuningTask(
    name='programmatic_optuna_nsgaii',
    param_space=param_space,
    instances=instances,
    seeds=[0, 1],
    budget_per_run=TRAIN_BUDGET,
    maximize=True,
    aggregator=np.mean,
)


## 2. Run the Optuna Study

This is the programmatic equivalent of the CLI flow, but it stays inside Python and returns the full trial history directly.


In [ ]:
tuner = ModelBasedTuner(
    task=task,
    max_trials=12,
    backend='optuna',
    optuna_sampler='tpe',
    seed=11,
    n_jobs=1,
)

best_config, history = tuner.run(eval_fn, verbose=False)

print(f'Collected {len(history)} trials.')
print('Best configuration keys:', sorted(best_config.keys())[:8], '...')


## 3. Tabulate and Export the Study

`history_to_dict(...)` gives a notebook-friendly representation, and the save helpers persist the same filtered history that the CLI post-processing uses.


In [ ]:
records = history_to_dict(history, task.param_space, include_raw=True)
summary = pd.DataFrame([
    {'trial_id': record['trial_id'], 'score': record['score'], **record['config']}
    for record in records
]).sort_values('score', ascending=False)

artifacts_dir = Path('results/notebooks/21_programmatic_tuning')
save_history_json(history, task.param_space, artifacts_dir / 'history.json', include_raw=True)
save_history_csv(history, task.param_space, artifacts_dir / 'history.csv', include_raw=True)

summary.head(8)


In [ ]:
best_so_far = np.maximum.accumulate(summary.sort_values('trial_id')['score'].to_numpy())

plt.figure(figsize=(8, 4.5))
plt.plot(best_so_far, 'o-', linewidth=2)
plt.title('Programmatic Optuna best-so-far trace')
plt.xlabel('Trial index')
plt.ylabel('Hypervolume')
plt.tight_layout()
plt.show()


## 4. Analyze the Real Optuna History

The same study history can be inspected directly in the notebook. This is the fastest way to see whether the search is converging and which parameters seem to matter.


In [ ]:
analysis_df = pd.DataFrame([
    {'trial_id': record['trial_id'], 'score': record['score'], **record['config']}
    for record in records
]).sort_values('trial_id').reset_index(drop=True)
analysis_df['best_so_far'] = analysis_df['score'].cummax()
analysis_df.head()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].plot(analysis_df['trial_id'], analysis_df['score'], 'o-', alpha=0.65, label='trial score')
axes[0].plot(analysis_df['trial_id'], analysis_df['best_so_far'], linewidth=2.5, label='best so far')
axes[0].set_title('Optuna convergence trace')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Hypervolume')
axes[0].legend()

axes[1].scatter(analysis_df['pop_size'], analysis_df['score'], s=70, alpha=0.75)
axes[1].set_title('Population size vs score')
axes[1].set_xlabel('pop_size')
axes[1].set_ylabel('Hypervolume')

axes[2].scatter(analysis_df['crossover_prob'], analysis_df['score'], s=70, alpha=0.75)
axes[2].set_title('Crossover probability vs score')
axes[2].set_xlabel('crossover_prob')
axes[2].set_ylabel('Hypervolume')

plt.tight_layout()
plt.show()


In [ ]:
top_slice = analysis_df.sort_values('score', ascending=False).head(5).reset_index(drop=True)
display_cols = ['trial_id', 'score', 'pop_size', 'crossover_prob', 'crossover_eta', 'mutation_eta']
top_slice[display_cols]


In [ ]:
numeric_cols = ['score', 'pop_size', 'crossover_prob', 'crossover_eta', 'mutation_eta']
corr = analysis_df[numeric_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.to_numpy(), cmap='coolwarm', vmin=-1.0, vmax=1.0)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr.index)), corr.index)
ax.set_title('Correlation matrix')
fig.colorbar(im, ax=ax, shrink=0.85)
plt.tight_layout()
plt.show()

corr['score'].sort_values(ascending=False)


## 5. Validate the Tuned Configuration

Tuning is only half of the workflow. We also want to check whether the tuned configuration beats a standard default on a held-out seed and a larger budget.


In [ ]:
VALIDATION_BUDGET = 3000
HOLDOUT_SEEDS = [77, 78]

tuned_cfg = config_from_assignment('nsgaii', dict(best_config))
default_cfg = NSGAIIConfig.default(pop_size=100, n_var=N_VAR)


def evaluate_cfg(cfg) -> float:
    scores = []
    for seed in HOLDOUT_SEEDS:
        result = optimize(
            'zdt1',
            algorithm='nsgaii',
            algorithm_config=cfg,
            max_evaluations=VALIDATION_BUDGET,
            seed=seed,
            n_var=N_VAR,
            engine='numpy',
        )
        scores.append(float(compute_hypervolume(result.F, REF_POINT)))
    return float(np.mean(scores))


tuned_score = evaluate_cfg(tuned_cfg)
default_score = evaluate_cfg(default_cfg)

pd.Series(
    {
        'tuned_mean_hv': tuned_score,
        'default_mean_hv': default_score,
        'delta': tuned_score - default_score,
    }
)


## Next Steps

- Continue with [33_optuna_tuning_advanced.ipynb](./33_optuna_tuning_advanced.ipynb) for persistent studies, pruning, BOHB-style runs, and multi-fidelity Optuna workflows.
